In [1]:
%pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 4.4 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.7 MB 4.4 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.7 MB 4.3 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/9.7 MB 4.3 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 4.5 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 4.5 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.7 MB 4.5 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 4.5 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.7 MB 4.5 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.7 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 4.5 MB/s  0:00:02

   ---------------------------------------- 0/2 [tzdata]
   -------------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import time
from datetime import datetime

# Each role maps to the set of (application, access_right) pairs it directly holds.
# These are the BASE permissions only
ROLE_PERMISSIONS = {
    "A": {
        "Money Market Instruments":     {"Read", "Write"},
        "Derivatives Trading":          {"Read", "Execute"},
        "Interest Instruments":         {"Read", "Execute"},
    },
    "B": {
        "Money Market Instruments":     {"Read", "Write", "Execute"},
        "Derivatives Trading":          {"Read", "Write", "Execute"},
        "Private Consumer Instruments": {"Read", "Write"},
    },
    "C": {
        "Investment Banking Suite":     {"Read", "Write", "Approve"},
        "Treasury Management":          {"Read", "Write", "Execute"},
    },
    "D": {
        "Loan Management":              {"Read", "Write", "Execute"},
    },
    "E": {
        "Risk Management System":       {"Read", "Write", "Execute", "Approve"},
    },
    "F": {
        "Corporate Trading Platform":   {"Read", "Write", "Execute", "Audit", "Report"},
    },
    "G": {
        "Share Management System":      {"Read", "Write"},
    },
    "H": {
        "E-Commerce Gateway":           {"Read", "Write", "Execute"},
    },
    "I": {
        "Office Banking Admin Panel":   {"Read", "Write", "Approve", "Audit"},
    },
}

# Role inheritance chain: each role listed inherits from the role before it.
ROLE_HIERARCHY = {
    "A": [],
    "B": ["A"],
    "C": ["B", "A"],
    "D": ["C", "B", "A"],
    "E": ["D", "C", "B", "A"],
    "F": ["E", "D", "C", "B", "A"],
    "G": [],
    "H": [],
    "I": [],
}

# Human-readable role metadata (Table 1)
ROLE_METADATA = {
    "A": {"function": "Financial Analyst",  "position": "Clerk"},
    "B": {"function": "Financial Analyst",  "position": "Junior Analyst"},
    "C": {"function": "Financial Analyst",  "position": "Senior Analyst"},
    "D": {"function": "Financial Analyst",  "position": "Specialist"},
    "E": {"function": "Financial Analyst",  "position": "Group Manager"},
    "F": {"function": "Financial Analyst",  "position": "Head of Division"},
    "G": {"function": "Share Technician",   "position": "Clerk"},
    "H": {"function": "Support E-Commerce", "position": "Junior"},
    "I": {"function": "Office Banking",     "position": "Head of Division"},
}

SESSION_DURATION = 3600
ACCESS_LOG_FILE  = "access_log.json"


def get_effective_permissions(role):
    """
    Returns the full set of (application -> access rights) for a given role
    by merging the role's own permissions with those inherited from parent roles.

    The merge is additive: if role B inherits from A, and both have entries for
    the same application, the resulting rights are the union of both sets.
    """
    effective = {}

    for ancestor in reversed(ROLE_HIERARCHY[role]):
        for app, rights in ROLE_PERMISSIONS.get(ancestor, {}).items():
            if app not in effective:
                effective[app] = set()
            effective[app] |= rights

    for app, rights in ROLE_PERMISSIONS.get(role, {}).items():
        if app not in effective:
            effective[app] = set()
        effective[app] |= rights

    return effective


def check_access(role, application, access_type):
    """
    Returns True if the given role (including inherited permissions) grants
    'access_type' on 'application', otherwise False.
    """
    effective = get_effective_permissions(role)
    return access_type in effective.get(application, set())

class User:
    """
    Represents a bank employee with:
      - A unique user ID
      - Up to four assigned roles
      - A single active role per session
      - Session start time used to enforce the 1-hour expiry window
    """

    MAX_ROLES = 4

    def __init__(self, user_id, roles):
        """
        Parameters
        ----------
        user_id : str
            Unique identifier for this user (e.g. 'User1').
        roles : list[str]
            The roles assigned to this user. At most MAX_ROLES roles are kept.
        """
        if not isinstance(roles, list) or len(roles) == 0:
            raise ValueError(f"User {user_id} must be assigned at least one role.")
        if len(roles) > self.MAX_ROLES:
            raise ValueError(
                f"User {user_id} cannot hold more than {self.MAX_ROLES} roles."
            )
        for r in roles:
            if r not in ROLE_PERMISSIONS:
                raise ValueError(f"Role '{r}' does not exist in the system.")

        self.user_id      = user_id
        self.roles        = roles          
        self.active_role  = None           
        self.session_start = None         

    def start_session(self, role):
        """
        Activates a session for this user under the specified role.
        The user must have that role assigned to them.

        Returns a status string indicating success or an error message.
        """
        if role not in self.roles:
            return f"ERROR: {self.user_id} cannot use role {role}."

        self.active_role   = role
        self.session_start = time.time()
        return f"{self.user_id} starts session with Role {role}"

    def end_session(self):
        """Clears the active session, forcing re-authentication for the next action."""
        self.active_role   = None
        self.session_start = None

    def is_session_valid(self):
        """
        Returns True if there is an active session that has not yet expired.
        A session is valid for SESSION_DURATION seconds from the start time.
        """
        if self.active_role is None or self.session_start is None:
            return False
        return (time.time() - self.session_start) < SESSION_DURATION

    def session_expired(self):
        """Returns True only if there WAS a session that has now timed out."""
        if self.active_role is None or self.session_start is None:
            return False
        return (time.time() - self.session_start) >= SESSION_DURATION

def log_access_attempt(user_id, application, access_type, result):
    """
    Appends a single access-attempt record to ACCESS_LOG_FILE (JSON lines format).

    Each record contains:
      timestamp   - human-readable UTC timestamp
      user        - the user ID
      application - the application that was requested
      access_type - the type of access requested (Read, Write, Execute, ...)
      result      - 'Granted' or 'Denied'
    """
    record = {
        "timestamp":    datetime.utcnow().strftime("%a, %d %b %Y %H:%M:%S GMT"),
        "user":         user_id,
        "application":  application,
        "access_type":  access_type,
        "result":       result,
    }

    existing = []
    try:
        with open(ACCESS_LOG_FILE, "r") as f:
            existing = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        pass

    existing.append(record)

    with open(ACCESS_LOG_FILE, "w") as f:
        json.dump(existing, f, indent=2)

def request_access(user, application, access_type):
    """
    Central access-control enforcement point.

    Steps:
      1. Verify the user has an active session.
      2. Check for session expiry.
      3. Verify the active role grants the requested access.
      4. Log the attempt regardless of outcome.
      5. Print a human-readable result line.

    Parameters
    ----------
    user        : User
    application : str  - name of the application being accessed
    access_type : str  - e.g. 'Read', 'Write', 'Execute', 'Approve', 'Audit', 'Report'
    """
    uid = user.user_id

    if user.active_role is None:
        msg = f"Access Denied: {uid} -> No active session."
        print(msg)
        log_access_attempt(uid, application, access_type, "Denied")
        return

    if user.session_expired():
        user.end_session()   # clean up state
        msg = f"ERROR: Session expired for {uid}."
        print(msg)
        log_access_attempt(uid, application, access_type, "Denied")
        return

    role = user.active_role
    if check_access(role, application, access_type):
        msg = f"Access Granted: {uid} -> {application} ({access_type})"
        print(msg)
        log_access_attempt(uid, application, access_type, "Granted")
    else:
        msg = f"Access Denied: {uid} -> {application} ({access_type})"
        print(msg)
        log_access_attempt(uid, application, access_type, "Denied")


In [3]:
def main():
    print("=" * 55)
    print("=== Creating Users ===")
    print("=" * 55)

    # User1 - Role B only (Junior Financial Analyst)
    user1 = User("User1", ["B"])
    print(f"Created {user1.user_id} with roles: {user1.roles}")

    # User2 - Role D (inherits C, B, A chain)
    user2 = User("User2", ["D"])
    print(f"Created {user2.user_id} with roles: {user2.roles}")

    # User3 - Multiple roles: A, G, H (up to 4 allowed)
    user3 = User("User3", ["A", "G", "H"])
    print(f"Created {user3.user_id} with roles: {user3.roles}")

    # User4 - Role F (top of the Financial Analyst chain)
    user4 = User("User4", ["F"])
    print(f"Created {user4.user_id} with roles: {user4.roles}")

    # User5 - Role I (Office Banking Head, no inheritance)
    user5 = User("User5", ["I"])
    print(f"Created {user5.user_id} with roles: {user5.roles}")

    
    print()
    print("=" * 55)
    print("=== Testing Role Inheritance ===")
    print("=" * 55)

    # User1 (Role B) inherits Role A permissions.
    # Role A has: Money Market Instruments (Read, Write)
    # Role B adds: Money Market Instruments (Execute) and more.
    # Expectation: User1 can Execute Money Market Instruments.
    print(user1.start_session("B"))
    request_access(user1, "Money Market Instruments", "Execute")   # Granted (B direct)
    request_access(user1, "Derivatives Trading",      "Write")     # Granted (B direct)
    request_access(user1, "Interest Instruments",     "Execute")   # Granted (inherited from A)

    # User2 (Role D) inherits D->C->B->A chain.
    # Should have access to everything from A through D.
    print()
    print(user2.start_session("D"))
    request_access(user2, "Money Market Instruments",     "Execute")  # Granted (via B)
    request_access(user2, "Interest Instruments",         "Read")     # Granted (via A)
    request_access(user2, "Investment Banking Suite",     "Approve")  # Granted (via C)
    request_access(user2, "Loan Management",              "Write")    # Granted (D direct)
    request_access(user2, "Risk Management System",       "Read")     # Denied (E not inherited by D)

    print()
    print("=" * 55)
    print("=== Testing Access Control ===")
    print("=" * 55)

    # User3 starts session with Role G (no inheritance)
    print(user3.start_session("G"))
    request_access(user3, "Share Management System",  "Read")    # Granted
    request_access(user3, "Share Management System",  "Write")   # Granted
    request_access(user3, "E-Commerce Gateway",       "Execute") # Denied (H, not active)

    # Switch User3 to Role H
    print()
    print(user3.start_session("H"))
    request_access(user3, "E-Commerce Gateway",       "Execute") # Granted
    request_access(user3, "Share Management System",  "Write")   # Denied (G, not active)

    # User4 (Role F) - top of hierarchy, has everything from A-F
    print()
    print(user4.start_session("F"))
    request_access(user4, "Corporate Trading Platform", "Audit")   # Granted (F direct)
    request_access(user4, "Risk Management System",     "Approve") # Granted (via E)
    request_access(user4, "Loan Management",            "Execute") # Granted (via D)
    request_access(user4, "Interest Instruments",       "Execute") # Granted (via A)
    request_access(user4, "Office Banking Admin Panel", "Audit")   # Denied (I, separate chain)

    # User5 (Role I) - isolated chain
    print()
    print(user5.start_session("I"))
    request_access(user5, "Office Banking Admin Panel", "Approve") # Granted
    request_access(user5, "Money Market Instruments",   "Read")    # Denied (outside I)

    print()
    print("=" * 55)
    print("=== Testing Error Handling ===")
    print("=" * 55)

    # 4a. User tries to start a session with a role they are not assigned
    print("-- Invalid role selection --")
    result = user1.start_session("F")   # User1 only has role B
    print(result)

    # 4b. User tries to access an application with no active session
    print()
    print("-- Access attempt with no session --")
    user_no_session = User("User6", ["C"])
    request_access(user_no_session, "Investment Banking Suite", "Read")

    # 4c. Active session but permission not granted for requested access type
    print()
    print("-- Permission denied (wrong access type) --")
    print(user1.start_session("B"))  # restore User1 session
    request_access(user1, "Loan Management", "Write")    # Role B has no path to D
    request_access(user1, "Corporate Trading Platform", "Audit")  # Role F out of reach

    # 4d. Simulate session expiry by manually backdating the session start
    print()
    print("-- Simulated session expiry --")
    user2.start_session("D")
    user2.session_start = time.time() - (SESSION_DURATION + 10)  # push start 1h+10s into past
    request_access(user2, "Loan Management", "Read")   # Should report session expired

    print()
    print("=" * 55)
    print("=== Audit Log (last 5 entries) ===")
    print("=" * 55)

    try:
        with open(ACCESS_LOG_FILE, "r") as f:
            log_entries = json.load(f)
        for entry in log_entries[-5:]:
            print(json.dumps(entry, indent=2))
    except (FileNotFoundError, json.JSONDecodeError):
        print("No log file found or log is empty.")


main()


=== Creating Users ===
Created User1 with roles: ['B']
Created User2 with roles: ['D']
Created User3 with roles: ['A', 'G', 'H']
Created User4 with roles: ['F']
Created User5 with roles: ['I']

=== Testing Role Inheritance ===
User1 starts session with Role B
Access Granted: User1 -> Money Market Instruments (Execute)
Access Granted: User1 -> Derivatives Trading (Write)
Access Granted: User1 -> Interest Instruments (Execute)

User2 starts session with Role D
Access Granted: User2 -> Money Market Instruments (Execute)
Access Granted: User2 -> Interest Instruments (Read)
Access Granted: User2 -> Investment Banking Suite (Approve)
Access Granted: User2 -> Loan Management (Write)
Access Denied: User2 -> Risk Management System (Read)

=== Testing Access Control ===
User3 starts session with Role G
Access Granted: User3 -> Share Management System (Read)
Access Granted: User3 -> Share Management System (Write)
Access Denied: User3 -> E-Commerce Gateway (Execute)

User3 starts session with Rol

C:\Users\willi\AppData\Local\Temp\ipykernel_40612\3223313485.py:185: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":    datetime.utcnow().strftime("%a, %d %b %Y %H:%M:%S GMT"),
